In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"E:\Imarticus BNGLR\FMCG Sales Forecsting 24-25\Dataset.csv")

df.head()

,Order_ID,Order_Date,Delivery_Date,Expected_Delivery,Delivery_Status,Payment_Mode,Payment_Status,Month_Name,Month_Number,Quarter,...,Quantity,Discount_Pct,Unit_Selling_Price,Gross_Value,Net_Sales_Value,Margin_Value,Target_Value,Is_Return,Gross_Margin_Pct,Achievement_Pct
0,ORD000525,2024-01-01,2024-01-06,2024-01-04,Delivered,UPI,Paid,January,1,Q1,...,32.4,2.8,94.71,3157.06,3068.60,709.88,3434.72,No,23.1,89.3
1,ORD000286,2024-01-01,2024-01-04,2024-01-05,Delivered,Credit,Paid,January,1,Q1,...,57.8,5.9,49.12,3017.16,2839.14,584.94,2862.87,No,20.6,99.2
2,ORD000310,2024-01-01,2024-01-06,2024-01-04,Delivered,UPI,Paid,January,1,Q1,...,19.0,0.8,163.98,3140.70,3115.62,769.12,3146.96,No,24.7,99.0
3,ORD000741,2024-01-01,2024-01-05,2024-01-04,Delivered,Cash,Overdue,January,1,Q1,...,15.0,3.7,65.35,1017.90,980.25,219.75,1071.61,No,22.4,91.5
4,ORD000605,2024-01-01,NaN,2024-01-05,Cancelled,Credit,Refunded,January,1,Q1,...,167.4,3.2,69.06,11942.32,11560.64,2638.22,12400.45,No,22.8,93.2


In [8]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Delivery_Date'] = pd.to_datetime(df['Delivery_Date'])
df['Expected_Delivery'] = pd.to_datetime(df['Expected_Delivery'])

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20366 entries, 0 to 20365
Data columns (total 48 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Order_ID            20366 non-null  str           
 1   Order_Date          20366 non-null  datetime64[us]
 2   Delivery_Date       13695 non-null  datetime64[us]
 3   Expected_Delivery   20366 non-null  datetime64[us]
 4   Delivery_Status     20366 non-null  str           
 5   Payment_Mode        20366 non-null  str           
 6   Payment_Status      20366 non-null  str           
 7   Month_Name          20366 non-null  str           
 8   Month_Number        20366 non-null  int64         
 9   Quarter             20366 non-null  str           
 10  Year                20366 non-null  int64         
 11  Calendar_YearMonth  20366 non-null  str           
 12  Distributor_Code    20366 non-null  str           
 13  Distributor_Name    20366 non-null  str           
 14  C

In [ ]:
#Delivery Delay in Days
df['Delivery_Delay_Days'] = (
    df['Delivery_Date']
    - df['Expected_Delivery']
).dt.days

In [ ]:
#Order Processing Time in Days
df['Order_Processing_Days'] = (
    df['Delivery_Date']
    - df['Order_Date']
).dt.days

In [12]:
df['Day_Of_Week'] = df['Order_Date'].dt.dayofweek

df['Week_Number'] = (
    df['Order_Date']
    .dt.isocalendar()
    .week
)

df['Month'] = df['Order_Date'].dt.month

df['Year'] = df['Order_Date'].dt.year

In [ ]:
#sort Values by Distributor Code and Order Date
df = df.sort_values(
    ['Distributor_Code','Order_Date']
)

In [19]:
df[['Distributor_Code','Order_Date']].head(20)

,Distributor_Code,Order_Date
10,DT1001,2024-01-01
11,DT1001,2024-01-01
15,DT1001,2024-01-01
22,DT1001,2024-01-02
36,DT1001,2024-01-02
46,DT1001,2024-01-03
50,DT1001,2024-01-03
53,DT1001,2024-01-03
62,DT1001,2024-01-03
85,DT1001,2024-01-04


In [22]:
if 'Sales_Lag_1' in df.columns:
    df.drop(columns=['Sales_Lag_1'], inplace=True)

In [23]:
#Target Variable
target = 'Net_Sales_Value'

In [24]:
#Creating Lag Features
lags = [1,3,7,14,21,30]

for lag in lags:
    df[f'Sales_Lag_{lag}'] = (
        df.groupby('Distributor_Code')[target]
          .shift(lag)
    )

In [25]:
df.columns

Index(['Order_ID', 'Order_Date', 'Delivery_Date', 'Expected_Delivery',
       'Delivery_Status', 'Payment_Mode', 'Payment_Status', 'Month_Name',
       'Month_Number', 'Quarter', 'Year', 'Calendar_YearMonth',
       'Distributor_Code', 'Distributor_Name', 'City', 'Zone', 'Tier',
       'Owner_Name', 'Phone', 'GST_Number', 'Credit_Limit', 'Years_Active',
       'DS_ID', 'DS_Name', 'DS_Type', 'Beat_Name', 'Beat_Key', 'Outlet_Name',
       'Outlet_Type', 'Channel_Type', 'SKU_Code', 'SKU_Name', 'Sub_Category',
       'Category', 'SBU', 'MRP', 'Cost_Price', 'Distributor_Price', 'Quantity',
       'Discount_Pct', 'Unit_Selling_Price', 'Gross_Value', 'Net_Sales_Value',
       'Margin_Value', 'Target_Value', 'Is_Return', 'Gross_Margin_Pct',
       'Achievement_Pct', 'Delivery_Delay_Days', 'Order_Processing_Days',
       'Day_Of_Week', 'Week_Number', 'Month', 'Sales_Lag_3', 'Sales_Lag_7',
       'Sales_Lag_14', 'Sales_Lag_21', 'Sales_Lag_30', 'Sales_Lag_1'],
      dtype='str')

In [26]:
rolling_windows = [3,7,14,21,30]

for window in rolling_windows:
    df[f'Rolling_{window}'] = (
        df.groupby('Distributor_Code')[target]
          .transform(
              lambda x: x.rolling(window).mean()
          )
    )

In [27]:
df.columns

Index(['Order_ID', 'Order_Date', 'Delivery_Date', 'Expected_Delivery',
       'Delivery_Status', 'Payment_Mode', 'Payment_Status', 'Month_Name',
       'Month_Number', 'Quarter', 'Year', 'Calendar_YearMonth',
       'Distributor_Code', 'Distributor_Name', 'City', 'Zone', 'Tier',
       'Owner_Name', 'Phone', 'GST_Number', 'Credit_Limit', 'Years_Active',
       'DS_ID', 'DS_Name', 'DS_Type', 'Beat_Name', 'Beat_Key', 'Outlet_Name',
       'Outlet_Type', 'Channel_Type', 'SKU_Code', 'SKU_Name', 'Sub_Category',
       'Category', 'SBU', 'MRP', 'Cost_Price', 'Distributor_Price', 'Quantity',
       'Discount_Pct', 'Unit_Selling_Price', 'Gross_Value', 'Net_Sales_Value',
       'Margin_Value', 'Target_Value', 'Is_Return', 'Gross_Margin_Pct',
       'Achievement_Pct', 'Delivery_Delay_Days', 'Order_Processing_Days',
       'Day_Of_Week', 'Week_Number', 'Month', 'Sales_Lag_3', 'Sales_Lag_7',
       'Sales_Lag_14', 'Sales_Lag_21', 'Sales_Lag_30', 'Sales_Lag_1',
       'Rolling_3', 'Rolling_7', 'Rol

In [28]:
#Sales Growth
df['Sales_Growth'] = (
    df.groupby('Distributor_Code')[target]
      .pct_change()
)

In [29]:
#Target Achievement Feature
#Target Gap
df['Target_Gap'] = (
    df['Target_Value']
    - df['Net_Sales_Value']
)

In [30]:
#Achievement Ratio
df['Achievement_Ratio'] = (
    df['Net_Sales_Value']
    / df['Target_Value']
)

In [31]:
df['Achievement_Ratio'] = np.where(
    df['Target_Value'] == 0,
    0,
    df['Net_Sales_Value'] / df['Target_Value']
)

In [33]:
nulls = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

print(nulls.head(20))

Delivery_Date            6671
Delivery_Delay_Days      6671
Order_Processing_Days    6671
Sales_Lag_30              450
Rolling_30                435
Sales_Lag_21              315
Rolling_21                300
Sales_Lag_14              210
Rolling_14                195
Sales_Lag_7               105
Rolling_7                  90
Sales_Lag_3                45
Rolling_3                  30
Sales_Lag_1                15
Sales_Growth               15
Order_Date                  0
Order_ID                    0
Expected_Delivery           0
Delivery_Status             0
GST_Number                  0
dtype: int64


In [34]:
#filling nulls with 0 for delay and processing time features
df['Delivery_Delay_Days'] = (
    df['Delivery_Delay_Days']
      .fillna(0)
)

df['Order_Processing_Days'] = (
    df['Order_Processing_Days']
      .fillna(0)
)

In [35]:
df.drop(columns=['Delivery_Date'], inplace=True)

In [36]:
df.isnull().sum().sort_values(ascending=False).head(20)

Sales_Lag_30         450
Rolling_30           435
Sales_Lag_21         315
Rolling_21           300
Sales_Lag_14         210
Rolling_14           195
Sales_Lag_7          105
Rolling_7             90
Sales_Lag_3           45
Rolling_3             30
Sales_Lag_1           15
Sales_Growth          15
Order_Date             0
Order_ID               0
Expected_Delivery      0
Delivery_Status        0
Payment_Mode           0
Payment_Status         0
GST_Number             0
Credit_Limit           0
dtype: int64

In [37]:
df = df.dropna()

In [41]:
df.columns

Index(['Order_Date', 'Expected_Delivery', 'Delivery_Status', 'Payment_Mode',
       'Payment_Status', 'Month_Name', 'Month_Number', 'Quarter', 'Year',
       'Calendar_YearMonth', 'Distributor_Code', 'Distributor_Name', 'City',
       'Zone', 'Tier', 'Credit_Limit', 'Years_Active', 'DS_ID', 'DS_Name',
       'DS_Type', 'Beat_Name', 'Beat_Key', 'Outlet_Name', 'Outlet_Type',
       'Channel_Type', 'SKU_Code', 'SKU_Name', 'Sub_Category', 'Category',
       'SBU', 'MRP', 'Cost_Price', 'Distributor_Price', 'Quantity',
       'Discount_Pct', 'Unit_Selling_Price', 'Gross_Value', 'Net_Sales_Value',
       'Margin_Value', 'Target_Value', 'Is_Return', 'Gross_Margin_Pct',
       'Achievement_Pct', 'Delivery_Delay_Days', 'Order_Processing_Days',
       'Day_Of_Week', 'Week_Number', 'Month', 'Sales_Lag_3', 'Sales_Lag_7',
       'Sales_Lag_14', 'Sales_Lag_21', 'Sales_Lag_30', 'Sales_Lag_1',
       'Rolling_3', 'Rolling_7', 'Rolling_14', 'Rolling_21', 'Rolling_30',
       'Sales_Growth', 'Target_Gap'

In [40]:
df = df.drop(columns=[
    'Order_ID',
    'GST_Number','Owner_Name', 'Phone',
       'GST_Number'
])

In [43]:
# Save
import os
os.makedirs("models", exist_ok=True)

df.to_csv("../models/processed_sales_data.csv", index=False)
print("Saved → ../models/processed_sales_data.csv")

Saved → ../models/processed_sales_data.csv
